# Parameter Golf — Kaggle Training Notebook

This notebook runs the Parameter Golf GPT training on Kaggle T4 x2 (or P100/single GPU).

**Setup required before running:**
1. Create a Kaggle Dataset containing `train_gpt_kaggle.py` (see KAGGLE_INSTRUCTIONS.md)
2. Attach that dataset to this notebook under Settings → Data
3. Select **GPU T4 x2** as your accelerator
4. Configure the variables in the cell below, then **Run All**

In [ ]:
# ============================================================
# CONFIGURATION — Edit these before running
# ============================================================

# Name of your Kaggle Dataset containing train_gpt_kaggle.py
# This is the slug shown in the dataset URL: kaggle.com/datasets/<YOUR_USERNAME>/<DATASET_NAME>
KAGGLE_DATASET_SLUG = "your-username/parameter-golf-scripts"  # ← CHANGE THIS

# Number of training data shards to download (each ~200MB, 100M tokens)
# More shards = more data = better model, but takes more disk & download time
# Kaggle has ~20GB disk. Recommended: 5-20 shards for experimentation
TRAIN_SHARDS = 20

# Number of GPUs to use (2 for T4 x2, 1 for P100 or single T4)
NUM_GPUS = 2

# Max training time in seconds (default 3600 = 1 hour)
# Kaggle sessions last up to 12 hours with GPU
MAX_WALLCLOCK = 3600

# Training iterations (default 20000, reduce for quick tests)
ITERATIONS = 20000

In [ ]:
# ============================================================
# CELL 1: Install dependencies & sanity check
# ============================================================
!pip install -q sentencepiece

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    cc = torch.cuda.get_device_capability(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"  GPU {i}: {name} (SM {cc[0]}.{cc[1]}, {mem:.1f} GB)")
print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")
print(f"Flash SDP available: {torch.cuda.get_device_capability(0) >= (8, 0)}")
print(f"\nUsing {NUM_GPUS} GPU(s) for training")

In [ ]:
# ============================================================
# CELL 2: Copy training script from your Kaggle Dataset
# ============================================================
import shutil
from pathlib import Path

# Kaggle mounts datasets at /kaggle/input/<dataset-name>/
dataset_name = KAGGLE_DATASET_SLUG.split("/")[-1]  # extract just the dataset name
src = Path(f"/kaggle/input/datasets/{KAGGLE_DATASET_SLUG}/train_gpt_kaggle.py")
dst = Path("/kaggle/working/train_gpt_kaggle.py")

if src.exists():
    shutil.copy2(src, dst)
    print(f"✓ Copied training script from {src}")
elif dst.exists():
    print(f"✓ Training script already at {dst}")
else:
    raise FileNotFoundError(
        f"Could not find train_gpt_kaggle.py at {src}\n"
        f"Make sure you created a Dataset with the file and set KAGGLE_DATASET_SLUG correctly.\n"
        f"Current value: '{KAGGLE_DATASET_SLUG}'"
    )

In [ ]:
# ============================================================
# CELL 3: Download training data from HuggingFace
# ============================================================
import json
import shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

REPO_ID = "willdepueoai/parameter-golf"
PREFIX = "datasets"
DATA_DIR = Path("/kaggle/working/data")
DATASETS_DIR = DATA_DIR / "datasets"
TOKENIZERS_DIR = DATA_DIR / "tokenizers"
VARIANT = "sp1024"
DATASET_NAME = f"fineweb10B_{VARIANT}"

# Fetch manifest to know shard counts and tokenizer paths
mf = Path(hf_hub_download(REPO_ID, "manifest.json", subfolder=PREFIX, repo_type="dataset"))
manifest = json.loads(mf.read_text())
ds_entry = next(x for x in manifest["datasets"] if x["name"] == DATASET_NAME)
tok_entry = next(x for x in manifest["tokenizers"] if x["name"] == ds_entry["tokenizer_name"])
val_shards = ds_entry["stats"]["files_val"]
max_train = ds_entry["stats"]["files_train"]

if TRAIN_SHARDS > max_train:
    raise ValueError(f"Requested {TRAIN_SHARDS} train shards but only {max_train} available.")

def dl(remote_path: str, dest: Path):
    """Download a single file from HF hub to a local destination."""
    if dest.exists():
        return
    rp = Path(remote_path)
    cached = Path(hf_hub_download(
        REPO_ID, rp.name,
        subfolder=rp.parent.as_posix() if rp.parent != Path(".") else None,
        repo_type="dataset"
    )).resolve()
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(cached, dest)

dest_ds = DATASETS_DIR / DATASET_NAME

# Download validation shards (always all of them)
print(f"Downloading {val_shards} validation shard(s)...")
for i in range(val_shards):
    dl(f"{PREFIX}/datasets/{DATASET_NAME}/fineweb_val_{i:06d}.bin",
       dest_ds / f"fineweb_val_{i:06d}.bin")

# Download training shards
print(f"Downloading {TRAIN_SHARDS} training shard(s) (of {max_train} available)...")
for i in range(TRAIN_SHARDS):
    dl(f"{PREFIX}/datasets/{DATASET_NAME}/fineweb_train_{i:06d}.bin",
       dest_ds / f"fineweb_train_{i:06d}.bin")
    if (i + 1) % 5 == 0 or i + 1 == TRAIN_SHARDS:
        print(f"  {i+1}/{TRAIN_SHARDS} shards downloaded")

# Download tokenizer file(s)
print("Downloading tokenizer...")
for key in ("model_path", "vocab_path", "path"):
    val = tok_entry.get(key)
    if val:
        rp = Path(val)
        dl(f"{PREFIX}/{val}", TOKENIZERS_DIR / rp.name)

# Verify
train_count = len(list(dest_ds.glob("fineweb_train_*.bin")))
val_count = len(list(dest_ds.glob("fineweb_val_*.bin")))
tok_files = list(TOKENIZERS_DIR.glob("*"))
print(f"\n✓ Data ready: {train_count} train shards, {val_count} val shards, {len(tok_files)} tokenizer file(s)")
print(f"  Dataset dir:   {dest_ds}")
print(f"  Tokenizer dir: {TOKENIZERS_DIR}")

In [ ]:
# ============================================================
# CELL 4: Run training
# ============================================================
# This runs the training script via torchrun for multi-GPU DDP.
# All output (stdout) appears below. Logs are also saved to /kaggle/working/logs/

import os
os.chdir("/kaggle/working")

env_overrides = f"MAX_WALLCLOCK_SECONDS={MAX_WALLCLOCK} ITERATIONS={ITERATIONS}"

if NUM_GPUS > 1:
    cmd = f"{env_overrides} torchrun --nproc_per_node={NUM_GPUS} /kaggle/working/train_gpt_kaggle.py"
else:
    cmd = f"{env_overrides} python /kaggle/working/train_gpt_kaggle.py"

print(f"Running: {cmd}\n{'='*80}")
!{cmd}

In [ ]:
# ============================================================
# CELL 5: Check results
# ============================================================
import os
from pathlib import Path

print("=== Output files ===")
for f in ["final_model.pt", "final_model.int8.ptz"]:
    p = Path(f"/kaggle/working/{f}")
    if p.exists():
        size_mb = p.stat().st_size / 1024 / 1024
        print(f"  ✓ {f}: {size_mb:.2f} MB")
    else:
        print(f"  ✗ {f}: not found")

# Show the last 30 lines of the training log
log_dir = Path("/kaggle/working/logs")
if log_dir.exists():
    log_files = sorted(log_dir.glob("*.txt"))
    if log_files:
        latest_log = log_files[-1]
        print(f"\n=== Last 30 lines of {latest_log.name} ===")
        lines = latest_log.read_text().splitlines()
        for line in lines[-30:]:
            print(line)
    else:
        print("\nNo log files found.")
else:
    print("\nNo logs directory found.")